# DMPSP Demo — Synthesis Route Planning

This notebook demonstrates DMPSP finding optimal synthesis routes for drug molecules.

**Requirements**: trained checkpoints in `checkpoints/` directory.
See `notebooks/01_train_kaggle.ipynb` or README for training instructions.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import torch
import yaml

from dmpsp import DMPSPPlanner, load_models, PlannerConfig

print(f'DMPSP ready. PyTorch {torch.__version__}. CUDA: {torch.cuda.is_available()}')

In [ ]:
# Load config and models
with open('../configs/model.yaml') as f:
    model_cfg = yaml.safe_load(f)

CHECKPOINT_DIR = '../checkpoints'  # adjust to your checkpoint directory
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

models = load_models(CHECKPOINT_DIR, model_cfg, device=DEVICE)
print('Models loaded:', list(models.keys()))

In [ ]:
# Create planner
planner = DMPSPPlanner(
    action_proposal=models['action_proposal'],
    world_model=models['world_model'],
    value_fn=models['value_fn'],
    encoder=models['encoder'],
    cfg=PlannerConfig(
        n_candidates=64,
        max_steps=10,
        device=DEVICE,
    ),
)

## Example 1: Aspirin — optimize for yield and cost

In [ ]:
route = planner.plan(
    target_smiles='CC(=O)Oc1ccccc1C(=O)O',   # aspirin
    objective_weights={
        'yield': 0.4,
        'cost': 0.4,
        'safety': 0.2,
    },
    max_steps=5,
)

print(f'Route: {route.n_steps} steps')
print(f'Yield:  {route.total_yield_fraction:.2%}')
print(f'Cost:   ${route.total_cost_usd:.2f}')
print(f'Time:   {route.planning_time_seconds:.2f}s')
print()
print('Objective scores:')
for name, score in sorted(route.objective_scores.items()):
    bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
    print(f'  {name:<20} {bar} {score:.3f}')

## Example 2: Same molecule — optimize for green chemistry (CDMO use case)

In [ ]:
route_green = planner.plan(
    target_smiles='CC(=O)Oc1ccccc1C(=O)O',
    objective_weights={
        'green_chem': 0.4,
        'manufacturability': 0.3,
        'safety': 0.3,
    },
    max_steps=5,
)

print('Green chemistry optimized route:')
print(f'  Green chem score:      {route_green.objective_scores["green_chem"]:.3f}')
print(f'  Manufacturability:     {route_green.objective_scores["manufacturability"]:.3f}')
print(f'  Safety:                {route_green.objective_scores["safety"]:.3f}')

## Example 3: Compare routes with different objective priorities

Same molecule, three different use cases — no retraining required.

In [ ]:
TARGET = 'CC(=O)Oc1ccccc1C(=O)O'

scenarios = {
    'Generic pharma (cost-optimize)': {
        'cost': 0.5, 'fto_risk': 0.3, 'yield': 0.2,
    },
    'Discovery (novelty-optimize)': {
        'novelty': 0.5, 'yield': 0.3, 'purity': 0.2,
    },
    'Sustainability (green-optimize)': {
        'green_chem': 0.5, 'safety': 0.3, 'manufacturability': 0.2,
    },
}

print(f'{'Scenario':<35} {'Steps':>6} {'Yield':>8} {'Cost':>8} {'Green':>8}')
print('-' * 70)

for scenario_name, weights in scenarios.items():
    r = planner.plan(TARGET, weights, max_steps=5)
    print(
        f'{scenario_name:<35} '
        f'{r.n_steps:>6} '
        f'{r.objective_scores["yield"]:>8.3f} '
        f'{r.objective_scores["cost"]:>8.3f} '
        f'{r.objective_scores["green_chem"]:>8.3f}'
    )

## Route step details

In [ ]:
print('Step-by-step breakdown:')
for i, (state, action) in enumerate(route.steps):
    print(f'\nStep {i+1}:')
    print(f'  Current molecule:  {state.current_smiles}')
    print(f'  Reaction class:    {action.reaction_class_id}')
    print(f'  Temperature:       {action.temperature_norm:.2f} (normalized)')
    print(f'  Solvent ID:        {action.solvent_id}')
    print(f'  Catalyst ID:       {action.catalyst_id}')

print(f'\nFinal product: {route.terminal_smiles}')